In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/voltpath/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Read bronze.sessions
bronze_sessions = spark.table(f"{catalog}.bronze.sessions")

# C1 — Standardise (applied to ALL rows before dedup, validation or joins)
standardized = bronze_sessions.select(
    # S1: session_id, charger_id, member_id → trim, upper; empty → NULL
    expr("nullif(trim(upper(session_id)), '')").alias("session_id"),
    expr("nullif(trim(upper(charger_id)), '')").alias("charger_id"),
    expr("nullif(trim(upper(member_id)), '')").alias("member_id"),
    # S2: status → trim, upper
    trim(upper(col("status"))).alias("status"),
    # S3: started_at, ended_at → valid only as yyyy-MM-dd HH:mm:ss or yyyy-MM-ddTHH:mm:ss
    expr(
        "CASE WHEN started_at IS NULL THEN NULL "
        "WHEN started_at LIKE '%T%' THEN try_to_timestamp(started_at, \"yyyy-MM-dd'T'HH:mm:ss\") "
        "ELSE try_to_timestamp(started_at, 'yyyy-MM-dd HH:mm:ss') END"
    ).alias("started_at_ts"),
    expr(
        "CASE WHEN ended_at IS NULL THEN NULL "
        "WHEN ended_at LIKE '%T%' THEN try_to_timestamp(ended_at, \"yyyy-MM-dd'T'HH:mm:ss\") "
        "ELSE try_to_timestamp(ended_at, 'yyyy-MM-dd HH:mm:ss') END"
    ).alias("ended_at_ts"),
    # S4: energy_kwh = meter.end_kwh − meter.start_kwh when both exist, else NULL
    when(
        col("meter.end_kwh").isNotNull() & col("meter.start_kwh").isNotNull(),
        col("meter.end_kwh").cast("decimal(10,3)") - col("meter.start_kwh").cast("decimal(10,3)")
    ).otherwise(lit(None).cast("decimal(10,3)")).alias("energy_kwh"),
    # S7: idle_minutes missing → 0
    when(col("idle_minutes").isNull(), lit(0))
    .otherwise(col("idle_minutes").cast("int")).alias("idle_minutes"),
    # S7: fault code → trim, upper
    expr("transform(faults, f -> named_struct('code', upper(trim(f.code)), 'at', f.at))").alias("faults_std"),
    # amount_paid from payment.amount_inr
    col("payment.amount_inr").cast("decimal(10,2)").alias("amount_paid"),
    # updated_at for dedup
    expr("try_to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss')").alias("updated_at_ts"),
    # Raw record (full Bronze row as JSON) for quarantine
    to_json(struct(*[col(c) for c in bronze_sessions.columns])).alias("raw_record"),
)

# S3: session_date = date of started_at
standardized = standardized.withColumn("session_date", col("started_at_ts").cast("date"))

# S5: duration_minutes = whole minutes from started_at to ended_at, rounded down; NULL if ended_at missing/invalid
standardized = standardized.withColumn(
    "duration_minutes",
    when(
        col("ended_at_ts").isNotNull() & col("started_at_ts").isNotNull(),
        floor((col("ended_at_ts").cast("long") - col("started_at_ts").cast("long")) / 60).cast("int")
    ).otherwise(lit(None).cast("int")),
)

# S6: time_band = PEAK if hour of started_at is 18–21, else OFF_PEAK
standardized = standardized.withColumn(
    "time_band",
    when(hour(col("started_at_ts")).between(18, 21), lit("PEAK")).otherwise(lit("OFF_PEAK")),
)

# C2 — De-duplicate: keep one version per session_id (latest updated_at)
w = Window.partitionBy("session_id").orderBy(col("updated_at_ts").desc())
deduped = (
    standardized.withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Bronze rows:", bronze_sessions.count())
print("Deduped rows:", deduped.count())

In [0]:
# C3 — Validate and quarantine
# Load reference data for R1 (silver.chargers) and R2 (silver.members)
charger_ids = [r.charger_id for r in spark.table(f"{catalog}.silver.chargers").select("charger_id").distinct().collect()]
member_ids = [r.member_id for r in spark.table(f"{catalog}.silver.members").select("member_id").distinct().collect()]

# Apply R1–R5 in order (lowest rule number wins)
validated = deduped.withColumn(
    "reject_reason",
    when(
        col("charger_id").isNull() | ~col("charger_id").isin(charger_ids),
        lit("R1: UNKNOWN_CHARGER"),
    )
    .when(
        col("member_id").isNotNull() & ~col("member_id").isin(member_ids),
        lit("R2: UNKNOWN_MEMBER"),
    )
    .when(col("started_at_ts").isNull(), lit("R3: INVALID_TIMESTAMP"))
    .when(
        col("status").isin("COMPLETED", "FAILED")
        & (col("ended_at_ts").isNull() | (col("ended_at_ts") <= col("started_at_ts"))),
        lit("R4: INVALID_DURATION"),
    )
    .when(
        (col("status") == "COMPLETED")
        & (col("energy_kwh").isNull() | (col("energy_kwh") < 0)),
        lit("R5: INVALID_METER"),
    )
    .otherwise(lit(None).cast("string")),
)

valid_sessions = validated.filter(col("reject_reason").isNull())
quarantined_sessions = validated.filter(col("reject_reason").isNotNull())


In [0]:
# C4 — Create target tables if not exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.sessions (
    session_id STRING,
    charger_id STRING,
    member_id STRING,
    started_at TIMESTAMP,
    ended_at TIMESTAMP,
    session_date DATE,
    status STRING,
    time_band STRING,
    energy_kwh DECIMAL(10,3),
    duration_minutes INT,
    idle_minutes INT,
    amount_paid DECIMAL(10,2),
    updated_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.session_faults (
    session_id STRING,
    fault_seq INT,
    fault_code STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.quarantine_sessions (
    session_id STRING,
    reject_reason STRING,
    updated_at TIMESTAMP,
    raw_record STRING,
    quarantined_at TIMESTAMP
) USING DELTA
""")

# --- Prepare DataFrames for MERGE ---

# Valid sessions → silver.sessions
silver_sessions_df = valid_sessions.select(
    col("session_id"), col("charger_id"), col("member_id"),
    col("started_at_ts").alias("started_at"),
    col("ended_at_ts").alias("ended_at"),
    col("session_date"), col("status"), col("time_band"),
    col("energy_kwh"), col("duration_minutes"), col("idle_minutes"),
    col("amount_paid"),
    col("updated_at_ts").alias("updated_at"),
)
silver_sessions_df.createOrReplaceTempView("vw_valid_sessions")

# Valid session faults → silver.session_faults (one row per fault, 1-based seq)
silver_faults_df = (
    valid_sessions
    .filter(col("faults_std").isNotNull() & (size(col("faults_std")) > 0))
    .select(col("session_id"), posexplode(col("faults_std")).alias("fault_seq", "fault"))
    .select(
        col("session_id"),
        (col("fault_seq") + 1).alias("fault_seq"),
        col("fault.code").alias("fault_code"),
    )
    .filter(col("fault_code").isNotNull())
)
silver_faults_df.createOrReplaceTempView("vw_valid_faults")

# Quarantined sessions → silver.quarantine_sessions
quarantine_df = quarantined_sessions.select(
    col("session_id"),
    col("reject_reason"),
    col("updated_at_ts").alias("updated_at"),
    col("raw_record"),
    current_timestamp().alias("quarantined_at"),
)
quarantine_df.createOrReplaceTempView("vw_quarantined")

# --- C5: Incremental MERGE (idempotent — re-running with no new data changes nothing) ---

# 1. MERGE valid sessions into silver.sessions
spark.sql(f"""
MERGE INTO {catalog}.silver.sessions AS t
USING vw_valid_sessions AS s
ON t.session_id = s.session_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

# 2. MERGE quarantined sessions into silver.quarantine_sessions
spark.sql(f"""
MERGE INTO {catalog}.silver.quarantine_sessions AS t
USING vw_quarantined AS s
ON t.session_id = s.session_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

# 3. Handle transitions: valid sessions removed from quarantine, quarantined removed from silver
spark.sql(f"DELETE FROM {catalog}.silver.quarantine_sessions WHERE session_id IN (SELECT session_id FROM vw_valid_sessions)")
spark.sql(f"DELETE FROM {catalog}.silver.sessions WHERE session_id IN (SELECT session_id FROM vw_quarantined)")

# 4. Handle silver.session_faults
# 4a. Delete faults for sessions that moved to quarantine
spark.sql(f"DELETE FROM {catalog}.silver.session_faults WHERE session_id IN (SELECT session_id FROM vw_quarantined)")

# 4b. Delete orphaned faults (in target for valid sessions but not in source)
# Delta does not support multi-column IN in DELETE or subqueries in MERGE conditions,
# so identify orphans via DataFrame joins and delete using a composite string key
orphaned_keys = (
    spark.table(f"{catalog}.silver.session_faults")
    .join(silver_sessions_df.select("session_id"), "session_id", "inner")
    .join(silver_faults_df, ["session_id", "fault_seq"], "left_anti")
    .select(concat_ws("|", col("session_id"), col("fault_seq").cast("string")).alias("fault_key"))
)
orphaned_keys.createOrReplaceTempView("vw_orphaned_keys")

spark.sql(f"""
DELETE FROM {catalog}.silver.session_faults
WHERE concat_ws('|', session_id, cast(fault_seq AS string)) IN (SELECT fault_key FROM vw_orphaned_keys)
""")

# 4c. MERGE new/updated faults
spark.sql(f"""
MERGE INTO {catalog}.silver.session_faults AS t
USING vw_valid_faults AS s
ON t.session_id = s.session_id AND t.fault_seq = s.fault_seq
WHEN MATCHED AND t.fault_code IS DISTINCT FROM s.fault_code THEN UPDATE SET t.fault_code = s.fault_code
WHEN NOT MATCHED THEN INSERT (session_id, fault_seq, fault_code) VALUES (s.session_id, s.fault_seq, s.fault_code)
""")

print("MERGE complete.")